# Train AI phát hiện bệnh lá trên Google Colab

Notebook này train 2 model cho hệ thống tưới cây:

- **Model tìm lá** (YOLO26n): khoanh từng chiếc lá cà chua, ớt chuông, khoai tây; bỏ lá cây khác. Ra `leaf_detector.onnx`.
- **Model bệnh** (EfficientNet-B0, 15 lớp): chấm bệnh từng chiếc lá. Ra `best.pt`.

**Trước khi chạy:** bấm **Runtime → Change runtime type → chọn GPU** (Colab Pro: L4 hoặc A100) → **Save**.
Sau đó chạy lần lượt từng ô (nút ▶), hoặc **Runtime → Run all**.

Kết quả được chép vào Google Drive, thư mục `MyDrive/tuoicay_ai`. Cách đưa về máy để chạy cùng Wokwi: xem ô cuối.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc

## 2. Kết nối Google Drive (để lưu model, máy Colab tắt là mất file)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Tải code và cài thư viện
Colab có sẵn PyTorch, torchvision, OpenCV, scikit-learn. Chỉ cài thêm phần còn thiếu.

In [ ]:
%cd /content
!rm -rf PlatformIO
!git clone -q --depth 1 https://github.com/HuongNguyenQuoc/PlatformIO.git
%cd /content/PlatformIO/Projects/heThongTuoiCayTT/ai
!pip install -q ultralytics onnxruntime onnxslim paho-mqtt
!mkdir -p logs

## 4. Tải dữ liệu (3–5 phút)

- **PlantVillage**: ảnh lá chụp trong phòng lab. Chỉ lấy thư mục cà chua, ớt chuông, khoai tây.
- **PlantDoc**: ảnh chụp ngoài vườn thật, dùng để train và chấm điểm thật (`test_real`).
- **PlantDoc Object Detection**: ảnh vườn có khung từng chiếc lá, dùng train model tìm lá.

In [ ]:
%cd /content/PlatformIO/Projects/heThongTuoiCayTT/ai
!mkdir -p data/raw
!git clone -q --filter=blob:none --no-checkout --depth 1 https://github.com/spMohanty/PlantVillage-Dataset.git data/raw/plantvillage
!git -C data/raw/plantvillage sparse-checkout set --no-cone '/raw/color/Tomato___*/' '/raw/color/Pepper,_bell___*/' '/raw/color/Potato___*/'
!git -C data/raw/plantvillage checkout -q
!git clone -q --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset.git data/raw/plantdoc
!git clone -q --depth 1 https://github.com/pratikkayal/PlantDoc-Object-Detection-Dataset.git data/raw/plantdoc_od/repo
!ls data/raw/plantvillage/raw/color   # phai co 15 thu muc: 2 Pepper, 3 Potato, 10 Tomato

## 5. Train model tìm lá (15–30 phút)

In [ ]:
import os
W = os.cpu_count()
!python prepare_detector_data.py
!python train_detector.py --batch 32 --workers {W}

## 6. Train model bệnh 15 lớp (10–20 phút)

In [ ]:
!python prepare_data.py
!python train.py --workers {W}

## 7. Chấm điểm

- `evaluate.py`: chấm từng ảnh/lá trên các tập test. **Điểm `test_real` (ảnh vườn thật) mới là điểm thật.**
- `evaluate.py --scenes`: chấm cả quy trình (tìm lá → chấm từng lá → bỏ phiếu) trên ảnh `test_real`.

Kết quả được lưu vào `logs/evaluate.log` và `logs/evaluate_scenes.log`.

In [ ]:
!python evaluate.py --workers {W} 2>&1 | tee logs/evaluate.log
!python evaluate.py --scenes --workers {W} 2>&1 | tee logs/evaluate_scenes.log

## 8. Thử phân tích 1 ảnh vườn thật của mỗi loại cây

In [ ]:
import glob, random
from IPython.display import Image, display
random.seed(1)
for crop in ["Tomato", "Pepper", "Potato"]:
    imgs = glob.glob(f"data/test_real/{crop}_*/*.jpg")
    if not imgs:
        print("Khong co anh test_real cho", crop)
        continue
    path = random.choice(imgs)
    print("=====", path)
    !python predict.py "{path}" --save /content/demo_{crop}.jpg
    display(Image(f"/content/demo_{crop}.jpg", width=400))

## 9. Lưu kết quả vào Google Drive

In [ ]:
OUT = "/content/drive/MyDrive/tuoicay_ai"
!mkdir -p {OUT}
!cp models/best.pt models/leaf_detector.onnx {OUT}/
!cp logs/*.log {OUT}/
!cd data && zip -qr {OUT}/test_real.zip test_real
!ls -la {OUT}

## 10. Đưa model về máy để chạy cùng Wokwi

1. Mở Google Drive → thư mục `tuoicay_ai` → tải về: `best.pt`, `leaf_detector.onnx`, `test_real.zip`.
2. Chép `best.pt` và `leaf_detector.onnx` vào `Projects\heThongTuoiCayTTi\models\` (tạo thư mục `models` nếu chưa có).
3. Giải nén `test_real.zip` vào `Projects\heThongTuoiCayTTi\data\` để có thư mục `ai\data	est_real` (ảnh mẫu cho `service.py`).
4. Chạy mô phỏng Wokwi (F1 → **Wokwi: Start Simulator**), đợi terminal Wokwi in `MQTT ok`.
5. Trong terminal VS Code:
   ```powershell
   cd Projects\heThongTuoiCayTTi
   .venv\Scripts\Activate.ps1
   python service.py --once
   ```
   LCD trên Wokwi hiện kết quả AI trong 20 giây.